In [1]:
import pandas as pd 
df=pd.read_csv("/Users/anishjain/Downloads/synthetic_rossmann_sales.csv")
df.head()

,Date,Store,DayOfWeek,Promo,Customers,Sales
0,2025-03-01,Store_1,6,0,110,1860
1,2025-03-02,Store_1,7,1,307,5092
2,2025-03-03,Store_1,1,0,67,1330
3,2025-03-04,Store_1,2,0,177,3919
4,2025-03-05,Store_1,3,0,79,1769


In [2]:
df.tail()

,Date,Store,DayOfWeek,Promo,Customers,Sales
145,2025-03-26,Store_5,3,0,240,3774
146,2025-03-27,Store_5,4,0,228,4993
147,2025-03-28,Store_5,5,0,62,1114
148,2025-03-29,Store_5,6,0,247,4572
149,2025-03-30,Store_5,7,1,118,2728


In [3]:
df.shape

(150, 6)

In [4]:
df.isnull().sum()

Date         0
Store        0
DayOfWeek    0
Promo        0
Customers    0
Sales        0
dtype: int64

In [5]:
df['Store'].unique()

array(['Store_1', 'Store_2', 'Store_3', 'Store_4', 'Store_5'],
      dtype=object)

In [6]:
df['Store'] = df['Store'].map({'Store_1': 1, 'Store_2': 2,'Store_3': 3, 'Store_4': 4,'Store_5': 5})
df.head()

,Date,Store,DayOfWeek,Promo,Customers,Sales
0,2025-03-01,1,6,0,110,1860
1,2025-03-02,1,7,1,307,5092
2,2025-03-03,1,1,0,67,1330
3,2025-03-04,1,2,0,177,3919
4,2025-03-05,1,3,0,79,1769


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['IsWeekend'] = df['Date'].dt.dayofweek >= 5  # Saturday or Sunday


# Drop 'Sales' (target) and 'Date' (non-numeric)
X = df.drop(['Sales', 'Date'], axis=1)

# One-hot encode the 'Store' column
X = pd.get_dummies(X, columns=['Store'], drop_first=True)

# Target
y = df['Sales']

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Build regression model
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(1)
])

# Compile model
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train model
model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=1)

# Predict and evaluate
y_pred = model.predict(X_test).flatten()

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"R² Score: {r2:.2f}")


Epoch 1/50


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13281849.0000 - mae: 3408.0510  
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 12959460.0000 - mae: 3362.9797
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13014200.0000 - mae: 3393.7805
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13054833.0000 - mae: 3385.1094
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 12868505.0000 - mae: 3363.8687
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13262661.0000 - mae: 3400.1733
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13125048.0000 - mae: 3394.0098 
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13693953.0000 - mae: 3458.5015 
Epoch 9/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13037815.0000 - mae: 3382.8906
Epoch 10/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13375596.0000 - mae: 3427.8672
Epoch 11/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 11793076.0000 - mae: 3194.7754
Epoch 12/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

In [14]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Linear Regression R²:", r2_score(y_test, y_pred_lr))


Linear Regression R²: 0.8590065008344134


In [15]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest R²:", r2_score(y_test, y_pred_rf))


Random Forest R²: 0.8510903967285395


In [17]:
from sklearn.svm import SVR

svr = SVR()
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)

print("SVR R²:", r2_score(y_test, y_pred_svr))


SVR R²: -0.0035632349403253993


In [18]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Linear Regression R²:", r2_score(y_test, y_pred_lr))


Linear Regression R²: 0.8590065008344134


In [19]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
df['WeekOfYear'] = df['Date'].dt.isocalendar().week


In [20]:
df = df.drop('Date', axis=1)


In [21]:
df.head()

,Store,DayOfWeek,Promo,Customers,Sales,Month,Day,IsWeekend,Year,WeekOfYear
0,1,5,0,110,1860,3,1,1,2025,9
1,1,6,1,307,5092,3,2,1,2025,9
2,1,0,0,67,1330,3,3,0,2025,10
3,1,1,0,177,3919,3,4,0,2025,10
4,1,2,0,79,1769,3,5,0,2025,10


In [22]:
from sklearn.ensemble import RandomForestRegressor


X = df.drop(['Sales'], axis=1)

# One-hot encode the 'Store' column
#X = pd.get_dummies(X, columns=['Store'], drop_first=True)

# Target
y = df['Sales']
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest R²:", r2_score(y_test, y_pred_rf))


Random Forest R²: 0.8510903967285395
